# 🍄 Mushroom Web Scraping & Data Collection Demo

This notebook demonstrates how to professionally collect high-quality mushroom image data using the **iNaturalist API**. 

### Why use an API instead of raw HTML scraping?
1. **Accuracy**: Data is verified by community experts.
2. **Stability**: APIs don't break when website layouts change.
3. **Ethical**: Respects rate limits and server resources.

In [ ]:
import os
import requests
import time
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

# Configuration
API_URL = "https://api.inaturalist.org/v1/observations"
SAVE_DIR = "demo_scrapped_images"

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

## 🔍 Step 1: Define Scraping Function
We fetch observations for a specific Taxon ID. Each observation contains photo URLs.

In [ ]:
def scrap_mushroom(taxon_id, species_name, target_count=5):
    print(f"🚀 Starting scrap for: {species_name}...")
    params = {
        "taxon_id": taxon_id,
        "quality_grade": "any",
        "per_page": target_count,
        "photos": "true"
    }
    
    response = requests.get(API_URL, params=params)
    data = response.json()
    results = data.get("results", [])
    
    downloaded = 0
    for obs in results:
        for photo in obs.get("photos", []):
            if downloaded >= target_count: break
            
            url = photo["url"].replace("square", "large")
            img_res = requests.get(url)
            
            if img_res.status_code == 200:
                img = Image.open(BytesIO(img_res.content))
                img_path = os.path.join(SAVE_DIR, f"{species_name}_{downloaded}.jpg")
                img.save(img_path)
                print(f"✅ Saved: {img_path}")
                downloaded += 1

    print(f"✨ Done! Collected {downloaded} images.")

## 🚀 Step 2: Run Scraping
Let's scrap some images of the **Fly Agaric (Amanita muscaria)**. Taxon ID: 48715.

In [ ]:
# TAXON_ID 48715 is for Fly Agaric
scrap_mushroom("48715", "Fly_Agaric", target_count=3)

## 🖼️ Step 3: Visualize Results
Displaying the scrapped images right here in the notebook.

In [ ]:
files = [f for f in os.listdir(SAVE_DIR) if "Fly_Agaric" in f]
plt.figure(figsize=(15, 5))

for i, f in enumerate(files[:3]):
    plt.subplot(1, 3, i+1)
    img = Image.open(os.path.join(SAVE_DIR, f))
    plt.imshow(img)
    plt.title(f)
    plt.axis("off")

plt.show()